# Getting Started with Sleight

This notebook walks through the core `sleight` API end-to-end:
1. Load a dataset
2. Build and train a model
3. Attack the model with FGSM and PGD
4. Evaluate robustness across epsilon values
5. Apply a defense (input transforms)
6. Visualize results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Load a Dataset

In [ ]:
from sleight.data import get_dataset

(x_train, y_train), (x_test, y_test) = get_dataset("mnist", one_hot=True)
y_test_int = np.argmax(y_test, axis=1)
print(f"Train: {x_train.shape}  Test: {x_test.shape}")

## 2. Build and Train a Model

In [ ]:
from sleight.models import get_mnist_cnn_model

model = get_mnist_cnn_model()
model.fit(x_train, y_train, epochs=3, batch_size=64, validation_split=0.1, verbose=1)

In [ ]:
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Clean test accuracy: {acc:.4f}")

## 3. Attack the Model

In [ ]:
from sleight.attacks import fgsm_attack, pgd_attack

x_sub = x_test[:200]
y_sub = y_test[:200]
y_sub_int = y_test_int[:200]

# FGSM
adv_fgsm = np.array(fgsm_attack(model, x_sub, y_sub, epsilon=0.15))
fgsm_acc = np.mean(np.argmax(model.predict(adv_fgsm, verbose=0), axis=1) == y_sub_int)
print(f"FGSM adversarial accuracy (eps=0.15): {fgsm_acc:.4f}")

# PGD
adv_pgd = np.array(pgd_attack(model, x_sub, y_sub, epsilon=0.15, alpha=0.01, num_iter=10))
pgd_acc = np.mean(np.argmax(model.predict(adv_pgd, verbose=0), axis=1) == y_sub_int)
print(f"PGD adversarial accuracy (eps=0.15):  {pgd_acc:.4f}")

In [ ]:
# Visualize
fig, axes = plt.subplots(3, 5, figsize=(12, 7))
for i in range(5):
    axes[0, i].imshow(x_sub[i, :, :, 0], cmap='gray')
    axes[0, i].set_title(f'True: {y_sub_int[i]}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(adv_fgsm[i, :, :, 0], cmap='gray')
    axes[1, i].set_title(f'FGSM: {np.argmax(model.predict(adv_fgsm[i:i+1], verbose=0))}')
    axes[1, i].axis('off')
    
    axes[2, i].imshow(adv_pgd[i, :, :, 0], cmap='gray')
    axes[2, i].set_title(f'PGD: {np.argmax(model.predict(adv_pgd[i:i+1], verbose=0))}')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('FGSM', fontsize=12)
axes[2, 0].set_ylabel('PGD', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Evaluate Robustness

In [ ]:
from sleight.evaluation import epsilon_sweep, plot_accuracy_vs_epsilon, plot_attack_comparison

epsilons = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

fgsm_results = epsilon_sweep(model, x_sub, y_sub_int, fgsm_attack, epsilons)
pgd_results = epsilon_sweep(model, x_sub, y_sub_int, pgd_attack, epsilons, alpha=0.01, num_iter=10)

plot_attack_comparison(
    {"FGSM": fgsm_results, "PGD": pgd_results},
    title="FGSM vs PGD on MNIST",
    show=True,
)

## 5. Apply a Defense

In [ ]:
from sleight.defenses import jpeg_compression, spatial_smoothing, bit_depth_reduction

# Apply defenses to adversarial images
defended_jpeg = jpeg_compression(adv_fgsm, quality=50)
defended_smooth = spatial_smoothing(adv_fgsm, kernel_size=3)
defended_bits = bit_depth_reduction(adv_fgsm, bits=3)

for name, defended in [("JPEG q=50", defended_jpeg), ("Smoothing k=3", defended_smooth), ("3-bit", defended_bits)]:
    acc = np.mean(np.argmax(model.predict(defended, verbose=0), axis=1) == y_sub_int)
    print(f"{name:15s} accuracy: {acc:.4f}")

print(f"{'No defense':15s} accuracy: {fgsm_acc:.4f}")

## 6. Summary

In this notebook we:
- Loaded MNIST with `sleight.data.get_dataset`
- Built a CNN with `sleight.models.get_mnist_cnn_model`
- Attacked it with FGSM and PGD from `sleight.attacks`
- Evaluated robustness with `sleight.evaluation`
- Applied input-transform defenses from `sleight.defenses`

For more, see the other notebooks or the [docs](../docs/README.md).